# Implementasi SCNN untuk Klasifikasi Citra Malware

Notebook Kaggle untuk melatih Simple Convolutional Neural Network (SCNN) pada Malimg Dataset.

## Metodologi
Preprocessing ? Augmentasi ? Training SCNN ? Evaluasi ? Confusion Matrix

In [ ]:
# Informasi perangkat keras untuk dokumentasi laporan
import platform
import shutil
import subprocess

print("Sistem operasi:", platform.platform())
print("Prosesor:", platform.processor())
try:
    import psutil
    print("RAM total (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
except ImportError:
    print("RAM: psutil tidak tersedia")

if gpus:
    for gpu in gpus:
        print("GPU:", gpu.name)
    try:
        gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True)
        print("GPU dan VRAM:", gpu_info.strip())
    except Exception:
        print("Detail VRAM tidak dapat dibaca")
else:
    print("GPU: tidak tersedia; training menggunakan CPU")


In [ ]:
"""CNN Malware Image Classification.

Script ini dapat dijalankan di Kaggle Notebook atau diubah menjadi cell notebook.
Dataset Malimg harus ditambahkan melalui Add Input.
"""

import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def find_dataset_root(base_dir="/kaggle/input"):
    """Find a directory whose immediate subdirectories contain images."""
    image_ext = {".png", ".jpg", ".jpeg", ".bmp"}
    candidates = []
    for root, dirs, files in os.walk(base_dir):
        current = Path(root)
        # Format dataset utama: root/train/<kelas>/*.png
        class_dirs = []
        for directory in dirs:
            child = current / directory
            if any(p.suffix.lower() in image_ext for p in child.rglob("*") if p.is_file()):
                class_dirs.append(child)
        if len(class_dirs) >= 2:
            candidates.append(current)
    if not candidates:
        raise FileNotFoundError("Dataset Malimg tidak ditemukan. Tambahkan dataset melalui Add Input.")
    # Pilih folder kelas yang paling spesifik, biasanya .../malimg_dataset/train.
    return sorted(candidates, key=lambda p: len(str(p)), reverse=True)[0]


def build_datasets(dataset_root):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_root,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_root,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
    )
    return train_ds, val_ds


def build_model(num_classes):
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
    ], name="augmentation")
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 1))
    x = augmentation(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 255)(x)
    for filters in (32, 64):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name="malware_cnn")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def plot_history(history):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history["accuracy"], label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].legend()
    axes[1].plot(history.history["loss"], label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


dataset_root = find_dataset_root()
print("Dataset:", dataset_root)
train_ds, val_ds = build_datasets(dataset_root)
class_names = train_ds.class_names
print("Jumlah kelas:", len(class_names))
print("Kelas:", class_names)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(autotune)
val_ds = val_ds.prefetch(autotune)
model = build_model(len(class_names))
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
plot_history(history)

y_true, y_pred = [], []
for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(18, 18))
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - CNN Malware Classification")
plt.tight_layout()
plt.show()

model.save("malware_cnn.keras")
print("Model tersimpan sebagai malware_cnn.keras")


In [ ]:
# Tabel ringkasan hasil untuk BAB IV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results_table = pd.DataFrame([{
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "Recall Macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "F1-Score Macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
}]).round(4)
results_table
